# Problem 1: Convolutions and Image Features
All of your code goes in `P1_features.py`. The Harris corner detector is provided in `utils.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import affine_transform, binary_erosion
from scipy.signal import convolve2d

from P1_features import conv2d, gaussian_kernel, sobel_gradients, patch_vectors, match_patches
from utils import detect_corners

os.makedirs("plots", exist_ok=True)
img = np.load("data/airport.npz")["image"] / 255.0
H, W = img.shape
print("image size:", img.shape)

## Filters
Your `conv2d` should match SciPy's `convolve2d` with zero padding, and your Gaussian mask should match the $3 \times 3$ example from lecture ($\sigma = 0.85$).

In [ ]:
rng = np.random.default_rng(0)
test_img, test_k = rng.normal(size=(30, 40)), rng.normal(size=(5, 3))
print("conv2d matches convolve2d:", np.allclose(conv2d(test_img, test_k), convolve2d(test_img, test_k, mode="same")))
print("16 * gaussian_kernel(3, 0.85) =")
print(np.round(16 * gaussian_kernel(3, 0.85), 2))

avg = conv2d(img, np.ones((5, 5)) / 25)
blur = conv2d(img, gaussian_kernel(13, 2.0))
Gx, Gy = sobel_gradients(img)

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
panels = [(img, "image"), (avg, "5x5 moving average"), (blur, "13x13 Gaussian, sigma = 2"),
          (Gx, "Gx"), (Gy, "Gy"), (np.hypot(Gx, Gy), "gradient magnitude")]
for ax, (im, title) in zip(axes.ravel(), panels):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.savefig("plots/p1_filters.png", dpi=120)
plt.show()

## Associativity
Smoothing and then differentiating is the same as convolving once with a combined filter (lecture: $F * (G * I) = (F * G) * I$). Away from the image border the two results agree.

In [ ]:
G = gaussian_kernel(13, 2.0)
Sx = np.array([[1., 0, -1], [2, 0, -2], [1, 0, -1]])
combined = convolve2d(G, Sx)    # full convolution of the two masks, 15 x 15
two_steps = conv2d(conv2d(img, G), Sx)
one_step = conv2d(img, combined)
print("combined mask size:", combined.shape)
print("max difference away from the border:", np.abs(two_steps - one_step)[10:-10, 10:-10].max())

## Matching across two views
The second view is the same image rotated by `angle` degrees about its center and shifted, with a little noise. Since we know the transform, we can check every match: a match is correct if the corner from view 1, mapped through the transform, lands within 2 pixels of the matched corner in view 2.

In [ ]:
def second_view(angle_deg, shift=(6.0, -9.0), noise=0.01, seed=0):
    """Returns the warped image, (A, b) with p2 = A p1 + b for (row, col) points, and a mask of the
    pixels at least 12 px inside the part covered by the original image."""
    th = np.deg2rad(angle_deg)
    A = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    c = np.array([H / 2, W / 2])
    b = c + np.asarray(shift) - A @ c
    out = affine_transform(img, A.T, offset=-A.T @ b, order=1, mode="nearest")
    out = out + noise * np.random.default_rng(seed).normal(size=out.shape)
    covered = affine_transform(np.ones_like(img), A.T, offset=-A.T @ b, order=0, mode="constant", cval=0.0) > 0.5
    return out, A, b, binary_erosion(covered, iterations=12)


def evaluate(angle_deg):
    img2, A, b, valid = second_view(angle_deg)
    c1, c2 = detect_corners(img), detect_corners(img2)
    c2 = c2[valid[c2[:, 0], c2[:, 1]]]    # drop corners on the replicated border
    # repeatability: fraction of view 1 corners that land in the valid part of view 2 and have a view 2 corner within 2 px
    p = c1 @ A.T + b
    q = np.round(p).astype(int)
    inside = (q[:, 0] >= 0) & (q[:, 0] < H) & (q[:, 1] >= 0) & (q[:, 1] < W)
    inside[inside] = valid[q[inside, 0], q[inside, 1]]
    dmin = np.linalg.norm(p[inside][:, None, :] - c2[None, :, :], axis=2).min(axis=1)
    repeat = np.mean(dmin < 2.0)
    matches, scores = match_patches(patch_vectors(img, c1), patch_vectors(img2, c2))
    correct = np.linalg.norm(p - c2[matches], axis=1) < 2.0
    return dict(img2=img2, c1=c1, c2=c2, matches=matches, correct=correct, repeat=repeat)


res = evaluate(8)
m, ok = res["matches"], res["correct"]
print(f"corners: {len(res['c1'])} (view 1), {len(res['c2'])} (view 2)")
print(f"correct matches: {ok.sum()} of {len(m)}")

canvas = np.hstack([img, res["img2"]])
fig, axes = plt.subplots(1, 2, figsize=(14, 3.2))
axes[0].imshow(img, cmap="gray")
axes[0].plot(res["c1"][:, 1], res["c1"][:, 0], "r+", ms=7)
axes[0].set_title(f"{len(res['c1'])} Harris corners, view 1")
axes[1].imshow(canvas, cmap="gray")
for i, (j, good) in enumerate(zip(m, ok)):
    r1, k1 = res["c1"][i]
    r2, k2 = res["c2"][j]
    axes[1].plot([k1, k2 + W], [r1, r2], "-", color="lime" if good else "red", lw=0.7)
axes[1].set_title(f"{ok.sum()} correct (green) and {(~ok).sum()} wrong (red) matches, 8 degree rotation")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("plots/p1_matches.png", dpi=120)
plt.show()

## Effect of rotation

In [ ]:
angles = [0, 5, 10, 15, 20, 30, 45, 60, 90]
rows = []
for a in angles:
    r = evaluate(a)
    rows.append((a, r["repeat"], r["correct"].sum()))
    print(f"angle {a:3d}: repeatability {r['repeat']:.3f}, correct matches {r['correct'].sum():3d} of {len(r['matches'])}")

rows = np.array(rows, dtype=float)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(rows[:, 0], rows[:, 1], "o-", color="tab:blue")
ax1.set_xlabel("rotation (deg)")
ax1.set_ylabel("corner repeatability", color="tab:blue")
ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx()
ax2.plot(rows[:, 0], rows[:, 2], "s-", color="tab:red")
ax2.set_ylabel("correct matches", color="tab:red")
ax1.set_title("Harris corners vs patch matching under rotation")
plt.tight_layout()
plt.savefig("plots/p1_rotation.png", dpi=120)
plt.show()